In [ ]:
import os
from dotenv import load_dotenv

# LangChain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

load_dotenv()

True

In [8]:
!pip install lxml beautifulsoup4 nest_asyncio

In [9]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from langchain_community.document_loaders import RecursiveUrlLoader

loader = RecursiveUrlLoader(
    url="https://debales.ai/",
    max_depth=100,
    extractor=lambda x: x  # default parser
)

docs = loader.load()

In [ ]:
len(docs)

217

In [73]:
docs[-1].metadata

{'source': 'https://debales.ai/terms-of-service',
 'loc': 'https://debales.ai/terms-of-service',
 'lastmod': '2026-04-06',
 'changefreq': 'yearly',
 'priority': '0.3'}

In [ ]:
unique_docs = {}
for i in docs:
    if i.metadata['source'].split('?')[0] not in unique_docs:
        unique_docs[i.metadata['source'].split('?')[0]] = i

In [ ]:
len(unique_docs)

130

In [55]:
docs[0].metadata['source']

'https://debales.ai'

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(list(unique_docs.keys()))
loader.requests_per_second = 3
clean_docs = loader.aload()


C:\Users\ADARSH\AppData\Local\Temp\ipykernel_54036\1534712975.py:5: LangChainDeprecationWarning: See API reference for updated usage: https://python.langchain.com/api_reference/community/document_loaders/langchain_community.document_loaders.web_base.WebBaseLoader.html
  clean_docs = loader.aload()
Fetching pages: 100%|##########| 130/130 [00:51<00:00,  2.54it/s]


11

In [63]:
len(clean_docs)

130

Document(metadata={'source': 'https://debales.ai/blog', 'title': 'Logistics Automation Blog | AI Insights for Freight Brokers & 3PLs | Debales | Debales AI', 'description': 'Expert insights on logistics automation, AI agents, TMS integration, and freight operations. 296+ articles for brokers, 3PLs, carriers.', 'language': 'en'}, page_content="Logistics Automation Blog | AI Insights for Freight Brokers & 3PLs | Debales | Debales AISolutions IntegrationsAI AgentsBlogCase StudiesSign UpSign InHomeBlogLogistics Automation Blog: AI Insights for Freight Brokers, 3PLs & CarriersExpert insights on logistics automation, AI agents, TMS integration, and freight operations.All blog postsCategoriesAll CategoriesLogistics & ShippingEcommerceAll blog postsMonday, 30 Mar 2026The Logistics AI Investment Trap: Why 80% Get Zero ROIOnly 20% of logistics AI investments deliver measurable ROI. BCG and MIT research reveals why most fail—and what the successful minority does differently.logistics AI ROIsupply

In [65]:
# 2. SPLIT INTO CHUNKS

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

split_docs = text_splitter.split_documents(clean_docs)
print(f"Split into {len(split_docs)} chunks")

Split into 1088 chunks


In [66]:
# 3. CREATE EMBEDDINGS

from langchain.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2"
)

C:\Users\ADARSH\AppData\Local\Temp\ipykernel_54036\3059366945.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3118.27it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [68]:
# 4. STORE IN VECTOR DB

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

vectorstore.persist()

print("Vector DB created successfully!")

Vector DB created successfully!


C:\Users\ADARSH\AppData\Local\Temp\ipykernel_54036\2178049201.py:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [70]:
# 5. RETRIEVER (for RAG)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Test query
query = "What does Debales AI do?"
results = retriever.invoke(query)

print("\nTop results:\n")
for i, doc in enumerate(results):
    print(f"{i+1}. {doc.page_content}\n")


Top results:

1. Debales AISolutions IntegrationsAI AgentsBlogCase StudiesSign UpSign InHomeBook DemoRequest a DemoTalk to a Debales.ai expert about your requirements, technology stack, and timeline. Complete the form and we'll make sure to reach out.Full Name *Work Email *Your email address will not be published. Our team will reach you as soon as possible.Phone NumberProject Details or Questions *How did you hear about us? *GoogleLinkedInChatGPTGeminiPerplexityOtherBy clicking "Yes", you agree to our Privacy Policy and Terms of Service. *SubmitPersonalized AI StrategyDiscover how our AI agents can be tailored to your specific workflows in logistics, manufacturing, or e-commerce.See Live ExamplesWitness our AI in action with real-world examples that demonstrate how we boost sales and streamline customer interactions.Get Clear ROI ProjectionsWe'll help you build a business case by discussing the potential return on investment for your company.Trusted ByYour Questions. Answered.Answers